# Lab type: debug
# Course: ML401 — MLOps & Model Deployment
# Lesson: Containerising ML Models
# Task: The Dockerfile and docker-compose.yml below each contain production issues. Identify all issues, explain the failure each causes, and write corrected versions.

## The broken Dockerfile

Your team has asked you to review this Dockerfile before it is used to build the production model serving image.
There are **4 issues** in this file. Identify each one before looking at the analysis.

```dockerfile
FROM python:latest

WORKDIR /app

COPY . .

RUN pip install -r requirements.txt

EXPOSE 8080

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8080"]
```

## Your Dockerfile analysis

**Issue 1:**
- Location in file:
- What is wrong:
- Failure mode in production:

**Issue 2:**
- Location in file:
- What is wrong:
- Failure mode in production:

**Issue 3:**
- Location in file:
- What is wrong:
- Failure mode in production:

**Issue 4:**
- Location in file:
- What is wrong:
- Failure mode in production:

<details>
<summary>🔑 Reveal answer — Issue 1: Unpinned base image</summary>

**What is wrong:** `FROM python:latest` resolves the tag at build time. An image built today and one built in six months may pull different Python versions, different base OS packages, and different system library versions.

**Failure mode:** Builds are not reproducible. A dependency that compiled fine against `python:latest` today may fail or behave differently after an upstream release. Compliance audits that require a known base image SHA will also fail.

**Fix:** `FROM python:3.11.9-slim` — pin the exact version and use the `slim` variant to minimise attack surface and image size.

</details>

<details>
<summary>🔑 Reveal answer — Issue 2: Layer cache defeated</summary>

**What is wrong:** `COPY . .` copies all source code into the image before `pip install`. Docker invalidates layers from the first changed layer downward, so every source code change (every deploy) forces `pip install` to rerun from scratch.

**Failure mode:** Build times balloon — a large `requirements.txt` adds minutes to every deploy even when no dependencies changed.

**Fix:** Split into two `COPY` instructions: `COPY requirements.txt .` → `RUN pip install ...` → `COPY . .`. Now pip only reruns when `requirements.txt` changes.

</details>

<details>
<summary>🔑 Reveal answer — Issue 3: Container runs as root</summary>

**What is wrong:** No `USER` instruction means the container process runs as root (UID 0).

**Failure mode:** If the application is compromised, the attacker has root access to the container filesystem. Most Kubernetes Pod Security Admission policies and OPA Gatekeeper rules will block root containers in production namespaces — the pod will not start.

**Fix:** `RUN useradd --create-home appuser` and `USER appuser` before the `CMD`.

</details>

<details>
<summary>🔑 Reveal answer — Issue 4: No HEALTHCHECK</summary>

**What is wrong:** No `HEALTHCHECK` directive is defined.

**Failure mode:** Docker and Kubernetes consider the container healthy as soon as the process starts. A model server that starts successfully but then fails to load the 2GB model artefact will accept live traffic and return errors (or silent wrong predictions) for every request.

**Fix:** Add a `HEALTHCHECK` with `--start-period=60s` (to allow model loading) that hits a `/health` endpoint: `HEALTHCHECK --interval=30s --timeout=10s --start-period=60s --retries=3 CMD curl -f http://localhost:8080/health || exit 1`.

</details>

## Write the corrected Dockerfile

Write a corrected Dockerfile in the cell below that fixes all four issues.
The corrected file should:
- Use a pinned Python base image
- Install dependencies before copying source code
- Run as a non-root user named `appuser`
- Include a HEALTHCHECK with appropriate startup period for a model that takes ~45 seconds to load

```dockerfile
# Write your corrected Dockerfile here:


```

<details>
<summary>🔑 Reveal corrected Dockerfile</summary>

```dockerfile
FROM python:3.11.9-slim

# Create non-root user
RUN useradd --create-home appuser
WORKDIR /home/appuser

# Install dependencies first (cached layer — only reruns when requirements.txt changes)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy source code after dependencies
COPY --chown=appuser:appuser . .

# Switch to non-root user
USER appuser

EXPOSE 8080

# Health check — start-period accounts for 45s model load time
HEALTHCHECK --interval=30s --timeout=10s --start-period=60s --retries=3 \\
  CMD curl -f http://localhost:8080/health || exit 1

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8080"]
```

</details>

## The broken docker-compose.yml

The following `docker-compose.yml` is used for local integration testing.
There are **3 issues** that will cause problems when this configuration is used as a reference for production deployment.

```yaml
version: '3.8'

services:
  model-api:
    build: .
    ports:
      - "8080:8080"
    environment:
      - MODEL_PATH=./models/churn_model.joblib
      - SECRET_KEY=dev-secret-key-do-not-use-in-production
    volumes:
      - ./models:/app/models
      - .:/app  # mount source code for hot reload
    restart: always
```

## Your docker-compose analysis

**Issue 1:**
- What is wrong:
- Why this is a problem if this configuration is referenced for production:

**Issue 2:**
- What is wrong:
- Why this is a problem if this configuration is referenced for production:

**Issue 3:**
- What is wrong:
- Why this is a problem if this configuration is referenced for production:

<details>
<summary>🔑 Reveal answer — docker-compose Issue 1: Hardcoded secret</summary>

**What is wrong:** `SECRET_KEY=dev-secret-key-do-not-use-in-production` is stored in plaintext in a file that will be committed to the repository.

**Production risk:** Secrets in source control are effectively public to anyone with repository access — and permanently visible in git history even after deletion. In production, secrets must be injected at runtime via a secrets manager (Kubernetes Secrets, AWS Secrets Manager, HashiCorp Vault), not from env vars in committed config files.

</details>

<details>
<summary>🔑 Reveal answer — docker-compose Issue 2: Source code volume mount</summary>

**What is wrong:** `. :/app` mounts the live source directory into the container, overriding the code baked into the image during build.

**Production risk:** The container is no longer running the tested, immutable image — it is running whatever happens to be on disk at runtime. If this pattern is copied into a staging or production config, untested code changes bypass the CI/CD pipeline entirely. The image is no longer the single source of truth.

</details>

<details>
<summary>🔑 Reveal answer — docker-compose Issue 3: restart: always without readiness gate</summary>

**What is wrong:** `restart: always` restarts the container on any exit, including startup failures (e.g., model artefact not found at `MODEL_PATH`).

**Production risk:** A container that fails to load the model will restart in a tight loop, consuming CPU and memory without ever surfacing the underlying error. In Kubernetes, this pattern is replaced by liveness and readiness probes with exponential backoff — the orchestrator stops routing traffic to unready containers and applies a backoff before restarting, making failures visible.

</details>

## Reflection

1. A colleague argues that because `docker-compose.yml` is only used locally, the issues above don't matter. What is the risk of this reasoning in practice?

2. If your model artefact is 2GB and must be included in the Docker image, how does this affect your build strategy? What alternatives exist to including the artefact directly in the image?

<details>
<summary>🔑 Reveal reflection answers</summary>

**1. Risk of "it's only used locally":** Config files treated as "local only" routinely get copied into staging or production configurations by colleagues under time pressure. The secret is also permanently in git history once committed — it cannot be un-leaked. Local dev configs are also a common vector for phishing and accidental exposure in screen-shares, public repos, and log files.

**2. 2GB model artefact in the image:** Every rebuild pushes and pulls 2GB+ over the network, slowing CI/CD to a crawl and bloating image registry storage. **Alternatives:**
- Pull the artefact from object storage (S3, GCS) at container startup, with the path passed via environment variable. The image stays small; the artefact is versioned independently.
- Use an init container (Kubernetes) to download the artefact to a shared volume before the model server starts.
- Use a model registry (MLflow, BentoML) that manages artefact storage and serving separately from the container image.

</details>